In [1]:
# import merged csv

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

df = pd.read_csv("wide_diff_all_data.csv")
df.head()

,subjectID,age_group,strata,Gender,DDisc_AUC_40K,Release,Acquisition,Age,3T_Full_MR_Compl,7T_Full_MR_Compl,...,Occipital_dki_mk,Orbital_dki_mk,PostParietal_dki_mk,SLF_L_dki_mk,SLF_R_dki_mk,SupFrontal_dki_mk,SupParietal_dki_mk,Temporal_dki_mk,UNC_L_dki_mk,UNC_R_dki_mk
0,100206,26-30,26-30_M,M,0.050000,S900,Q11,26-30,True,False,...,1.101557,1.084557,1.053842,1.107496,1.110725,1.112234,1.026142,1.056521,0.995955,0.988601
1,100307,26-30,26-30_F,F,0.311459,Q1,Q01,26-30,True,False,...,1.006925,1.045562,0.987310,1.008259,1.024426,0.984244,0.928578,0.976830,0.895127,0.915798
2,100610,26-30,26-30_M,M,0.868750,S900,Q08,26-30,True,True,...,0.996652,1.105037,1.005991,1.009783,1.057179,1.042984,0.986284,1.009254,0.956977,0.977282
3,101006,31-35,31-35_F,F,0.783073,S500,Q06,31-35,True,False,...,0.931849,1.074894,0.951102,1.011284,1.040509,1.017018,0.981339,0.962131,0.965962,1.004965
4,101107,22-25,22-25_M,M,0.584375,S500,Q06,22-25,True,False,...,0.971370,1.107099,0.975676,0.999982,1.025620,1.027364,0.963823,0.960659,0.917482,0.953508


In [3]:

dki_cols = [col for col in df.columns if "dki_mk" in col]

target = pd.to_numeric(df["DDisc_AUC_40K"], errors="coerce")

correlations = pd.DataFrame({
    "column": dki_cols,
    "correlation": [
        pd.to_numeric(df[col], errors="coerce").corr(target)
        for col in dki_cols
    ]
}).sort_values("correlation", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)

correlations

,column,correlation
0,SupFrontal_dki_mk,-0.074378
1,ARC_L_dki_mk,-0.068484
2,Temporal_dki_mk,-0.064737
3,ARC_R_dki_mk,-0.060489
4,Motor_dki_mk,-0.059246
5,CGC_R_dki_mk,-0.055815
6,CST_R_dki_mk,-0.054667
7,SLF_R_dki_mk,-0.054186
8,IFO_R_dki_mk,-0.046887
9,CGC_L_dki_mk,-0.046307


In [4]:
print(df.columns)

Index(['subjectID', 'age_group', 'strata', 'Gender', 'DDisc_AUC_40K',
       'Release', 'Acquisition', 'Age', '3T_Full_MR_Compl', '7T_Full_MR_Compl',
       ...
       'Occipital_dki_mk', 'Orbital_dki_mk', 'PostParietal_dki_mk',
       'SLF_L_dki_mk', 'SLF_R_dki_mk', 'SupFrontal_dki_mk',
       'SupParietal_dki_mk', 'Temporal_dki_mk', 'UNC_L_dki_mk',
       'UNC_R_dki_mk'],
      dtype='object', length=107)


In [5]:
df["DDisc_AUC_40K"].isna().sum()

0

In [6]:
df.loc[df["DDisc_AUC_40K"].isna(), "subjectID"].nunique()

0

In [7]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

X = df[diffusion_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.408902,0.408902
1,PC2,0.103779,0.512681
2,PC3,0.068946,0.581627
3,PC4,0.054600,0.636227
4,PC5,0.032579,0.668806
5,PC6,0.027890,0.696696
6,PC7,0.024486,0.721182
7,PC8,0.021412,0.742594
8,PC9,0.018252,0.760846
9,PC10,0.015659,0.776505


In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Diffusion-only columns
diffusion_cols = [col for col in df.columns if "dki_" in col]

X = df[diffusion_cols].copy()

# Make sure everything is numeric
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

# Standardize
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.408902,0.408902
1,PC2,0.103779,0.512681
2,PC3,0.068946,0.581627
3,PC4,0.054600,0.636227
4,PC5,0.032579,0.668806
5,PC6,0.027890,0.696696
6,PC7,0.024486,0.721182
7,PC8,0.021412,0.742594
8,PC9,0.018252,0.760846
9,PC10,0.015659,0.776505


In [9]:
# Add age and sex back in as covariates
pca_cov_df = pd.concat(
    [
        pca_df,
        df[["age_group", "Gender"]].copy()
    ],
    axis=1
)

pca_cov_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC18,PC19,PC20,PC21,PC22,PC23,PC24,PC25,age_group,Gender
0,6.417739,2.753540,4.468664,2.798375,1.761349,1.743399,-1.655228,0.937363,-1.440674,0.048492,...,0.364610,-0.312299,1.079550,1.809428,-0.258620,-2.648596,-0.512922,0.245447,26-30,M
1,-6.300341,1.496958,-3.630617,1.152875,1.410573,2.070698,-0.582060,1.449655,1.801464,1.656467,...,0.238596,-1.144116,0.611303,0.082469,1.087177,-1.919819,-0.195404,-0.913575,26-30,F
2,-8.668389,0.236776,6.631749,-0.232100,0.124864,-0.162685,-0.232459,1.816967,1.225516,0.761905,...,-0.022972,0.867962,-0.479318,0.120221,-0.119325,0.158382,-0.459894,0.455271,26-30,M
3,-7.412201,-0.683953,2.797153,-4.508080,-3.085005,-1.718423,1.539881,0.934043,-0.559129,-0.757529,...,0.317390,-0.985771,-0.539406,-0.180611,-0.597791,1.157718,-0.028858,0.596184,31-35,F
4,-3.433597,2.099954,-1.765756,-2.858150,-0.362014,1.717135,-0.785887,-1.700770,-2.300731,0.226133,...,-0.934665,-0.338880,0.227167,-0.563858,0.599312,0.009472,0.267358,-0.798865,22-25,M


In [10]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm

# PCA on diffusion variables only
diffusion_cols = [col for col in df.columns if "dki_" in col]

X_diff = df[diffusion_cols].copy()
X_diff = X_diff.apply(pd.to_numeric, errors="coerce").fillna(X_diff.mean())

X_scaled = StandardScaler().fit_transform(X_diff)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

# Covariates: DD score, age, sex
cov_df = df[["DDisc_AUC_40K", "age_group", "Gender"]].copy()

# Make outcome numeric
cov_df["DDisc_AUC_40K"] = pd.to_numeric(cov_df["DDisc_AUC_40K"], errors="coerce")

# One-hot encode covariates
cov_df = pd.get_dummies(cov_df, columns=["age_group", "Gender"], drop_first=True)

# Combine and keep only complete cases
model_df = pd.concat([cov_df, pca_df], axis=1).dropna()

y = model_df["DDisc_AUC_40K"]
X = model_df.drop(columns=["DDisc_AUC_40K"])
X = sm.add_constant(X)

# Force numeric types
y = y.astype(float)
X = X.astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     1.796
Date:                Wed, 22 Jul 2026   Prob (F-statistic):            0.00676
Time:                        20:14:11   Log-Likelihood:                -86.762
No. Observations:                 691   AIC:                             233.5
Df Residuals:                     661   BIC:                             369.7
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4820      0.029     

In [11]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

X = df[diffusion_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.mean())

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

explained_variance = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(25)],
    "ExplainedVariance": pca.explained_variance_ratio_,
    "CumulativeVariance": pca.explained_variance_ratio_.cumsum()
})

explained_variance

,PC,ExplainedVariance,CumulativeVariance
0,PC1,0.408902,0.408902
1,PC2,0.103779,0.512681
2,PC3,0.068946,0.581627
3,PC4,0.054600,0.636227
4,PC5,0.032579,0.668806
5,PC6,0.027890,0.696696
6,PC7,0.024486,0.721182
7,PC8,0.021412,0.742594
8,PC9,0.018252,0.760846
9,PC10,0.015659,0.776505


In [12]:
import pandas as pd
import statsmodels.api as sm

pc_cols = [col for col in pca_df.columns if col.startswith("PC")]

reg_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df[pc_cols]
    ],
    axis=1
)

reg_df["DDisc_AUC_40K"] = pd.to_numeric(reg_df["DDisc_AUC_40K"], errors="coerce")
reg_df = pd.get_dummies(reg_df, columns=["age_group", "Gender"], drop_first=True)
reg_df = reg_df.dropna()

y = reg_df["DDisc_AUC_40K"].astype(float)
X = reg_df.drop(columns=["DDisc_AUC_40K"]).astype(float)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     1.796
Date:                Wed, 22 Jul 2026   Prob (F-statistic):            0.00676
Time:                        20:14:11   Log-Likelihood:                -86.762
No. Observations:                 691   AIC:                             233.5
Df Residuals:                     661   BIC:                             369.7
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4820      0.029     

In [13]:
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1) Keep only diffusion columns with the substrings you specified
pc_substrings = ["dki_fa", "dki_md", "dki_mk", "dki_awf"]

diffusion_cols = [
    col for col in df.columns
    if any(sub in col for sub in pc_substrings)
]

print("Number of diffusion columns:", len(diffusion_cols))
print("First 10 diffusion columns:", diffusion_cols[:10])

# 2) Build PCA only from those columns
X_diff = df[diffusion_cols].copy()
X_diff = X_diff.apply(pd.to_numeric, errors="coerce")
X_diff = X_diff.fillna(X_diff.mean())

X_scaled = StandardScaler().fit_transform(X_diff)

pca = PCA(n_components=25, random_state=42)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=[f"PC{i+1}" for i in range(25)],
    index=df.index
)

# 3) Confirm loadings only involve diffusion variables
loadings = pd.DataFrame(
    pca.components_.T,
    index=diffusion_cols,
    columns=[f"PC{i+1}" for i in range(25)]
)

print("\nTop loadings for PC19:")
display(
    loadings["PC19"]
    .abs()
    .sort_values(ascending=False)
    .head(15)
    .to_frame("abs_loading")
    .join(loadings["PC19"].to_frame("loading"))
)

# 4) Regression: DD score on PCs + age + sex
reg_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df
    ],
    axis=1
)

reg_df["DDisc_AUC_40K"] = pd.to_numeric(reg_df["DDisc_AUC_40K"], errors="coerce")
reg_df = pd.get_dummies(reg_df, columns=["age_group", "Gender"], drop_first=True)
reg_df = reg_df.dropna()

y = reg_df["DDisc_AUC_40K"].astype(float)
X = reg_df.drop(columns=["DDisc_AUC_40K"]).astype(float)
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())

Number of diffusion columns: 96
First 10 diffusion columns: ['ARC_L_dki_awf', 'ARC_R_dki_awf', 'ATR_L_dki_awf', 'ATR_R_dki_awf', 'AntFrontal_dki_awf', 'CGC_L_dki_awf', 'CGC_R_dki_awf', 'CST_L_dki_awf', 'CST_R_dki_awf', 'IFO_L_dki_awf']

Top loadings for PC19:


,abs_loading,loading
CST_R_dki_mk,0.349484,0.349484
PostParietal_dki_fa,0.300073,0.300073
ILF_R_dki_fa,0.258051,0.258051
CST_L_dki_mk,0.241175,0.241175
UNC_R_dki_fa,0.226078,0.226078
PostParietal_dki_md,0.214847,-0.214847
SupParietal_dki_fa,0.191053,0.191053
CST_R_dki_fa,0.185631,-0.185631
IFO_L_dki_awf,0.182523,-0.182523
CST_L_dki_fa,0.163335,-0.163335


                            OLS Regression Results                            
Dep. Variable:          DDisc_AUC_40K   R-squared:                       0.073
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     1.796
Date:                Wed, 22 Jul 2026   Prob (F-statistic):            0.00676
Time:                        20:14:11   Log-Likelihood:                -86.762
No. Observations:                 691   AIC:                             233.5
Df Residuals:                     661   BIC:                             369.7
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4820      0.029     

In [14]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm

# Build modeling table from your existing PCA scores + covariates
model_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df
    ],
    axis=1
)

model_df["DDisc_AUC_40K"] = pd.to_numeric(model_df["DDisc_AUC_40K"], errors="coerce")
model_df = pd.get_dummies(model_df, columns=["age_group", "Gender"], drop_first=True)
model_df = model_df.dropna()

y = model_df["DDisc_AUC_40K"].astype(float)
X = model_df.drop(columns=["DDisc_AUC_40K"]).astype(float)

# Nested cross-validation setup
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": np.logspace(-3, 3, 25)
}

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

# Outer CV predictions
y_pred = cross_val_predict(search, X, y, cv=outer_cv, n_jobs=-1)

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print(f"Nested CV R2: {r2:.4f}")
print(f"Nested CV RMSE: {rmse:.4f}")

Nested CV R2: 0.0018
Nested CV RMSE: 0.2847


In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

model_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df
    ],
    axis=1
)

model_df["DDisc_AUC_40K"] = pd.to_numeric(model_df["DDisc_AUC_40K"], errors="coerce")
model_df = pd.get_dummies(model_df, columns=["age_group", "Gender"], drop_first=True)
model_df = model_df.dropna()

y = model_df["DDisc_AUC_40K"].astype(float)
X = model_df.drop(columns=["DDisc_AUC_40K"]).astype(float)

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(max_iter=10000))
])

param_grid = {
    "model__alpha": np.logspace(-3, 2, 20),
    "model__l1_ratio": np.linspace(0.1, 0.9, 9)
}

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

y_pred = cross_val_predict(search, X, y, cv=outer_cv, n_jobs=-1)

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print(f"Nested CV R2: {r2:.4f}")
print(f"Nested CV RMSE: {rmse:.4f}")

Nested CV R2: -0.0129
Nested CV RMSE: 0.2868


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

class FirstNPCsSelector(BaseEstimator, TransformerMixin):
    def __init__(self, n_pcs=5):
        self.n_pcs = n_pcs

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.iloc[:, :self.n_pcs]

model_df = pd.concat(
    [
        df[["DDisc_AUC_40K", "age_group", "Gender"]],
        pca_df
    ],
    axis=1
)

model_df["DDisc_AUC_40K"] = pd.to_numeric(model_df["DDisc_AUC_40K"], errors="coerce")
model_df = pd.get_dummies(model_df, columns=["age_group", "Gender"], drop_first=True)
model_df = model_df.dropna()

y = model_df["DDisc_AUC_40K"].astype(float)

covariate_cols = [col for col in model_df.columns if col.startswith("age_group_") or col.startswith("Gender_")]
pc_cols = [col for col in model_df.columns if col.startswith("PC")]

X_cov = model_df[covariate_cols]
X_pcs = model_df[pc_cols]

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("pc_select", FirstNPCsSelector()),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(max_iter=10000))
])

param_grid = {
    "pc_select__n_pcs": [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25],
    "model__alpha": np.logspace(-3, 2, 20),
    "model__l1_ratio": np.linspace(0.1, 0.9, 9)
}

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

y_pred = cross_val_predict(search, X_pcs.join(X_cov), y, cv=outer_cv, n_jobs=-1)

rmse = np.sqrt(mean_squared_error(y, y_pred))
r2 = r2_score(y, y_pred)

print(f"Nested CV R2: {r2:.4f}")
print(f"Nested CV RMSE: {rmse:.4f}")